# ChromaFs Quickstart

This notebook demonstrates how to populate a ChromaDB collection with the ChromaFs schema and query it using `ChromaFsBackend`.

**What we'll build:**
- A virtual filesystem with two pages: `data/paris.md` and `data/vietnam.md`
- A path tree document that lets `ChromaFsBackend` bootstrap instantly
- Content chunks for each page, queryable via `ls`, `read`, and `grep`

**ChromaFs schema at a glance:**
```
collection
├── __path_tree__          ← one JSON doc listing all slugs + RBAC metadata
├── data/paris.md::0       ← chunk 0  (metadata: page_slug, chunk_index)
├── data/paris.md::1       ← chunk 1
├── data/vietnam.md::0
└── data/vietnam.md::1
```

## 1. Start ChromaDB

Run a local ChromaDB server via Docker. Data is persisted in `./chroma-data`.

In [3]:
! docker run -d -v ./chroma-data:/data -p 8000:8000 chromadb/chroma

e5152b76160fc6598fb23a3cb992d0f3344173f9510a90d492d074c03d12596e


In [4]:
! uv pip install openai

Using Python 3.13.11 environment at: /home/kiennd/Code/Projects/deepagents-chromafs/.venv
Resolved 16 packages in 30ms                                         
Installed 1 package in 39ms                                 
 + openai==2.33.0


## 2. Imports & config

In [5]:
import json
import os

import chromadb
from openai import OpenAI

HOST = "localhost"
PORT = 8000
COLLECTION_NAME = "docs"

OPENAI_API_KEY = "" # fill OPENAI_API_KEY in here
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIM = 3072  # default output dimension for text-embedding-3-large

## 3. Embedding setup

`text-embedding-3-large` produces 3072-dimensional vectors by default. The `embed()` helper accepts a batch of texts and returns the corresponding embedding matrix in a single API call.

In [6]:
openai_client = OpenAI(api_key=OPENAI_API_KEY)


def embed(texts: list[str]) -> list[list[float]]:
    """Return embeddings for a batch of texts using EMBEDDING_MODEL."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

## 4. Connect to ChromaDB

We disable ChromaDB's built-in embedding function (`embedding_function=None`) because embeddings are generated externally by OpenAI and passed explicitly on every `upsert` call.

In [7]:
client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,
)
print(f"Collection '{collection.name}' ready — {collection.count()} documents")

Collection 'docs' ready — 0 documents


## 5. Prepare data

Define all content before inserting. Each page is represented as a list of text chunks that will be stored separately in ChromaDB and reassembled on read.

### 5.1 Path tree

The path tree is a flat JSON map of every slug in the collection. `ChromaFsBackend` fetches this once on startup to build an in-memory directory index — no further network calls are needed for `ls` or `glob`.

- `isPublic: true` — visible to all users regardless of group membership
- `groups` — list of group names that may access the path when `isPublic` is `false`

In [8]:
path_tree = {
    "data/paris.md":   {"isPublic": True, "groups": []},
    "data/vietnam.md": {"isPublic": True, "groups": []},
}

### 5.2 Page chunks

Each page's content is split into chunks. Chunks are stored as separate ChromaDB documents with two required metadata fields:
- `page_slug` — identifies which page the chunk belongs to (must match a key in the path tree)
- `chunk_index` — integer ordering used to reassemble the full page content

In [9]:
chunks_paris = [
    "Paris, the capital of France, is globally renowned as the 'City of Light' — a city where art, history, and everyday life blend seamlessly.",
    "At its heart stands the Eiffel Tower, the most iconic architectural landmark in the city, drawing millions of visitors from around the world each year.",
    "Beyond the monuments, Parisian cafe culture and freshly baked croissants define the rhythm of daily life for the city's residents.",
    "A short walk from any arrondissement leads to the Louvre Museum, home to priceless masterpieces including the legendary Mona Lisa.",
    "The Seine River ties it all together, winding through the city and framing its historic bridges in a landscape that has inspired artists and lovers for centuries.",
]

chunks_vietnam = [
    "Vietnam is a Southeast Asian country of remarkable contrasts — its long coastline, lush mountains, and river deltas together form one of the most diverse landscapes in the region.",
    "This natural richness is mirrored in Vietnamese cuisine, celebrated worldwide for its fresh ingredients and balance of flavors, with Pho standing as the country's most iconic dish.",
    "Nowhere is Vietnam's natural grandeur more dramatic than Ha Long Bay, a UNESCO World Heritage site where thousands of limestone islands rise from emerald waters.",
    "The people who call this land home have long been admired for their resilience and the genuine warmth with which they welcome strangers.",
    "That spirit is most alive in Hanoi and Ho Chi Minh City, two great metropolises where ancient temple districts sit side by side with gleaming modern towers.",
]

## 6. Insert into ChromaDB

### 6.1 Insert path tree

The path tree is stored as a single document with the reserved ID `__path_tree__`.

Two precautions to prevent it from polluting search results:
- **Zero-vector embedding** — ensures the document never wins a cosine similarity match
- **`_system: true` metadata** — allows filtering it out from custom `where_document` queries

In [10]:
collection.upsert(
    ids=["__path_tree__"],
    documents=[json.dumps(path_tree)],
    embeddings=[[0.0] * EMBEDDING_DIM],
    metadatas=[{"_system": True}],
)
print("Path tree inserted")

Path tree inserted


### 6.2 Insert page chunks

For each page, all chunks are embedded in a single batched API call to minimise latency and cost, then upserted into ChromaDB together.

In [11]:
def build_chunk_records(slug: str, chunks: list[str]) -> dict:
    """Build ids / documents / metadatas / embeddings for a list of text chunks."""
    ids = [f"{slug}::{i}" for i in range(len(chunks))]
    metadatas = [{"page_slug": slug, "chunk_index": i} for i in range(len(chunks))]
    embeddings = embed(chunks)
    return {"ids": ids, "documents": chunks, "metadatas": metadatas, "embeddings": embeddings}


for slug, chunks in [
    ("data/paris.md",   chunks_paris),
    ("data/vietnam.md", chunks_vietnam),
]:
    records = build_chunk_records(slug, chunks)
    collection.upsert(**records)
    print(f"Inserted {len(records['ids'])} chunks for {slug}")

Inserted 5 chunks for data/paris.md
Inserted 5 chunks for data/vietnam.md


## 7. Verify with ChromaFsBackend

Create a `ChromaFsBackend` instance and exercise the read API. The backend bootstraps its in-memory tree from `__path_tree__` at construction time — subsequent `ls`, `read`, and `grep` calls need no extra network round-trips beyond fetching page content.

In [12]:
from deepagents_chromafs import ChromaFsBackend

backend = ChromaFsBackend(collection)

# Directory listing
print("ls /:", backend.ls("/").entries)
print("ls /data:", backend.ls("/data").entries)

# Read a full page
print("\n--- /data/paris.md ---")
print(backend.read("/data/paris.md").file_data["content"])

# Grep across all pages
print("\ngrep 'UNESCO':")
for match in backend.grep("UNESCO").matches:
    print(f"  {match['path']}:{match['line']}: {match['text']}")

ls /: [{'path': '/data', 'is_dir': True}]
ls /data: [{'path': '/data/paris.md', 'is_dir': False}, {'path': '/data/vietnam.md', 'is_dir': False}]

--- /data/paris.md ---
Paris, the capital of France, is globally renowned as the 'City of Light' — a city where art, history, and everyday life blend seamlessly.
At its heart stands the Eiffel Tower, the most iconic architectural landmark in the city, drawing millions of visitors from around the world each year.
Beyond the monuments, Parisian cafe culture and freshly baked croissants define the rhythm of daily life for the city's residents.
A short walk from any arrondissement leads to the Louvre Museum, home to priceless masterpieces including the legendary Mona Lisa.
The Seine River ties it all together, winding through the city and framing its historic bridges in a landscape that has inspired artists and lovers for centuries.

grep 'UNESCO':
  /data/vietnam.md:3: Nowhere is Vietnam's natural grandeur more dramatic than Ha Long Bay, a UNESC